# VibeSight — Yüz İfadesi Modeli Eğitimi (Kaggle GPU)

**Veri:** FER2013 görüntüleri + FER+ (Microsoft, MIT) oy etiketleri · **Model:** ConvNeXt-Tiny (ImageNet-22k ön-eğitimli)
· **Çıktı:** kalibre edilmiş `emotion.onnx` + `emotion.json`

### Başlamadan önce (sağ paneldeki *Notebook options / Settings*)
1. **Accelerator → GPU T4 x2** (veya P100)
2. **Internet → On** (telefon doğrulaması gerekir; repo ve FER+ etiketleri indirilecek)
3. **+ Add Input** → arama: `fer2013` → **`deadskull7/fer2013`** veri setini ekleyin (içinde `fer2013.csv` olmalı)
4. Sağ üstte **Save Version → Save & Run All (Commit)** — arka planda çalışır, sekmeyi kapatabilirsiniz. Süre: T4'te yaklaşık 20–25 dakika.
   (*Run All* ile etkileşimli çalıştırırsanız oturum boşta kalınca kapanır ve çıktılar silinir.)

In [ ]:
REPO_URL = "https://github.com/erenkadiroglu/VibeSight.git"
MODEL = "convnext_tiny.fb_in22k"
EPOCHS = 20

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf vibesight && git clone --depth 1 $REPO_URL vibesight
!pip install -q "timm>=1.0" onnx onnxscript onnxruntime

In [ ]:
import glob
candidates = glob.glob("/kaggle/input/**/fer2013.csv", recursive=True) + glob.glob("/kaggle/input/**/icml_face_data.csv", recursive=True)
assert candidates, "fer2013.csv bulunamadı — + Add Input ile 'deadskull7/fer2013' veri setini ekleyin."
FER = candidates[0]
print("Kullanılan dosya:", FER)

In [ ]:
%cd /kaggle/working/vibesight/training
!python train.py --fer2013 "$FER" --model $MODEL --epochs $EPOCHS --batch 128 --workers 4 --out /kaggle/working/run

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open("/kaggle/working/run/metrics.json"))
print(json.dumps({k: m[k] for k in ("test", "test_uncalibrated", "temperature", "per_class_recall")}, indent=2, ensure_ascii=False))
display(Image("/kaggle/working/run/confusion.png", width=640))

In [ ]:
# İndirilecek paket: emotion.onnx + emotion.json + metrikler (sağ panel → Output → vibesight-emotion.zip)
!cd /kaggle/working/run && zip -q -j /kaggle/working/vibesight-emotion.zip emotion.onnx emotion.json metrics.json confusion.png history.csv
!ls -lh /kaggle/working/vibesight-emotion.zip

### Sonraki adım
`vibesight-emotion.zip` içindeki **`emotion.onnx`** ve **`emotion.json`** dosyalarını bilgisayarınızda `backend/models/` klasörüne kopyalayın
(taban modelin üzerine yazar) ve sunucuyu yeniden başlatın. Arayüzün sağ üstündeki model bilgisinde yeni model adı görünür.